# Layer compression by input year (SAE-TCAV Work 8, addendum)

Addendum outside the pre-declared plan (plan v1 addendum 2026-11-26). Pooled multi-year evaluation strata mix a year-tracking leading direction into the early-layer spectrum; this notebook splits the saved evaluation rows by input year. Per-year probe AUROC is not computed: fitting-row arrays were not saved.

The inputs are the per-row arrays saved by the two layer-compression runs (`layer_arrays.npz`). Nothing is fitted again: the notebook recomputes the layer geometry of each year group from the saved embeddings, gradients and probabilities, with one patient bootstrap per group.

## 1. Imports and constants
One cell holds the paths, the names of the saved builds and strata, and the bootstrap settings. The computation runs on the CPU (numpy eigen-decompositions of 192-dimensional rows; minutes, not hours), so no GPU is used.

In [1]:
import json
import platform
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

import flchain_probe as probe
import layer_compression as lc
import prospective_audit as pa
import tcav_geometry as geometry

T0 = time.monotonic()
STARTED_UTC = datetime.now(timezone.utc).isoformat()
STEP_SECONDS = {}

SUS_DIR = Path("analysis_outputs/sus_layer_compression")
SUS_OUT = SUS_DIR / "by_year"
FLCHAIN_DIR = Path("analysis_outputs/flchain_layer_compression")
FLCHAIN_OUT = FLCHAIN_DIR / "by_year"
FLCHAIN_CSV = Path("data/flchain/free_light_chain_mortality.csv")
CODE_FILES = [Path(name) for name in ("layer_compression.py", "flchain_probe.py", "prospective_audit.py", "tcav_geometry.py")]
VERSIONED_PACKAGES = ("numpy", "pandas", "scikit-learn", "torch", "matplotlib", "threadpoolctl")

# (array prefix in layer_arrays.npz, build, scaling, stratum name in the main layer_curves.csv)
SUS_SERIES = (
    ("same_year_consistent", "same_year_last_3_2011", "consistent", "input_years_2011_2013"),
    ("same_year_historical", "same_year_last_3_2011", "historical", "input_years_2011_2013"),
    ("prospective_consistent", "prospective_last_3_2011", "consistent", "input_years_2011_2013"),
)
FLCHAIN_BUILD = "horizon_1095d"
FLCHAIN_SCALING = "consistent"
FLCHAIN_HORIZON_DAYS = probe.PRIMARY_HORIZON_DAYS
FLCHAIN_STRATA = ("1997_eval", "1998-2003")
FLCHAIN_YEAR_GROUPS = {1998: "1998", 1999: "1999", 2000: "2000", 2001: "2001", 2002: "2002-2003", 2003: "2002-2003"}

DRAWS = 500
BOOTSTRAP_SEED = pa.BOOTSTRAP_SEED
DIRECTION_SEED = probe.DIRECTION_SEED
BOOTSTRAP_THREADS = lc.BOOTSTRAP_THREAD_LIMIT
SANITY_TOLERANCE = 1e-6
GRADIENT_KINDS = ("probability", "log_odds")
SANITY_METRICS = (
    ("target", "raw", "embedding_pr"),
    ("target", "standardized", "embedding_pr"),
    ("target", "raw", "probability_resultant_length"),
    ("target", "standardized", "probability_resultant_length"),
)
# (file name, title, y label, metric, chart, standard error metric or None)
FIGURE_SPECS = (
    ("spectrum_embedding_pr_raw", "Embedding participation ratio, raw chart", "participation ratio",
     "embedding_pr", "raw", None),
    ("spectrum_embedding_pr_standardized", "Embedding participation ratio, standardized chart",
     "participation ratio", "embedding_pr", "standardized", None),
    ("spectrum_top1_share_raw", "Top-1 principal share of variance, raw chart", "share of variance",
     "top1_var_share", "raw", None),
    ("gradient_probability_resultant_length_raw", "Gradient resultant length R (probability), raw chart",
     "R", "probability_resultant_length", "raw", None),
    ("random_isotropic_interior_share_raw",
     "Random isotropic interior share, raw chart (bars: 1 SE of the directions)", "interior share",
     "random_isotropic_interior_share", "raw", "random_isotropic_interior_se"),
)

for folder in (SUS_OUT, FLCHAIN_OUT, SUS_OUT / "figures", FLCHAIN_OUT / "figures"):
    folder.mkdir(parents=True, exist_ok=True)
print(f"Draws {DRAWS} | seed {BOOTSTRAP_SEED} | BLAS threads in bootstrap {BOOTSTRAP_THREADS} | device cpu")


Draws 500 | seed 20261106 | BLAS threads in bootstrap 1 | device cpu


## 2. Load the saved arrays
Each series is one SUS build or one FLChain evaluation stratum. The FLChain input year of each saved row comes from the same CSV, target and chronological split as the Work 6b notebook; the subject id of every saved row is checked against that split. Each row of a series is grouped by its input year (FLChain: the stratum `1997_eval` is one group, `1998-2003` is split by year, with 2002 and 2003 together, as in the Work 6b strata).

In [2]:
step_start = time.monotonic()


def sus_series_list(arrays):
    """One series per SUS build, with the rows and arrays saved in layer_arrays.npz."""
    series_list = []
    for prefix, build, scaling, stratum in SUS_SERIES:
        key = f"{prefix}__"
        years = arrays[key + "year"].astype(np.int64)
        series_list.append({
            "dataset": "sus", "group_column": "input_year",
            "build": build, "scaling": scaling, "stratum": stratum,
            "years": years, "group_label": years.astype(str),
            "patient_ids": arrays[key + "patient_id"].astype(str),
            "probability": arrays[key + "probability"].astype(np.float64),
            "deaths": arrays[key + "label"].astype(np.int64),
            "target": arrays[key + "target_embedding"].astype(np.float64),
            "feature_mean": arrays[key + "feature_mean_embedding"].astype(np.float64),
            "gradient": arrays[key + "target_gradient"].astype(np.float64),
            "scale_target": arrays[key + "scale_target"].astype(np.float64),
            "scale_feature_mean": arrays[key + "scale_feature_mean"].astype(np.float64),
        })
    return series_list


def flchain_series_list(arrays, kept, y):
    """One series per FLChain evaluation stratum; the sample year of each saved row comes from the split."""
    rows = arrays["eval_row_index"].astype(np.int64)
    if not np.array_equal(kept["subject_id"].to_numpy()[rows], arrays["subject_id"]):
        raise ValueError("subject ids of the saved rows do not match the FLChain split")
    stratum_of_row = arrays["eval_stratum"].astype(str)
    years = kept["sample.yr"].to_numpy(dtype=np.int64)[rows]
    if set(years[stratum_of_row == "1997_eval"].tolist()) != {1997}:
        raise ValueError("the 1997_eval stratum must hold only 1997 rows")
    group_of_row = np.array([
        stratum if stratum == "1997_eval" else FLCHAIN_YEAR_GROUPS[int(year)]
        for stratum, year in zip(stratum_of_row, years)])
    series_list = []
    for stratum in FLCHAIN_STRATA:
        mask = stratum_of_row == stratum
        series_list.append({
            "dataset": "flchain", "group_column": "sample_year_group",
            "build": FLCHAIN_BUILD, "scaling": FLCHAIN_SCALING, "stratum": stratum,
            "years": years[mask], "group_label": group_of_row[mask],
            "patient_ids": arrays["subject_id"][mask].astype(str),
            "probability": arrays["probability"][mask].astype(np.float64),
            "deaths": y[rows][mask].astype(np.int64),
            "target": arrays["target_embedding"][mask].astype(np.float64),
            "feature_mean": arrays["feature_mean_embedding"][mask].astype(np.float64),
            "gradient": arrays["target_gradient"][mask].astype(np.float64),
            "scale_target": arrays["scale_target"].astype(np.float64),
            "scale_feature_mean": arrays["scale_feature_mean"].astype(np.float64),
        })
    return series_list


sus_arrays = np.load(SUS_DIR / "layer_arrays.npz")
flchain_arrays = np.load(FLCHAIN_DIR / "layer_arrays.npz")
flchain_frame = probe.load_flchain_dataframe(FLCHAIN_CSV, expected_sha256=probe.EXPECTED_CSV_SHA256)
flchain_kept, flchain_y = probe.build_flchain_target(flchain_frame, FLCHAIN_HORIZON_DAYS)

sus_all = sus_series_list(sus_arrays)
flchain_all = flchain_series_list(flchain_arrays, flchain_kept, flchain_y)
all_series = sus_all + flchain_all
LAYER_COUNT = sus_all[0]["target"].shape[1]
LAST_LAYER = LAYER_COUNT - 1
for series in all_series:
    if series["target"].shape[1:] != (LAYER_COUNT, 192):
        raise ValueError(f"unexpected embedding shape in {series['build']} {series['stratum']}")

group_counts = pd.DataFrame([
    {"dataset": series["dataset"], "build": series["build"], "scaling": series["scaling"],
     "stratum": series["stratum"], "group": group,
     "rows": int(np.sum(series["group_label"] == group)),
     "deaths": int(series["deaths"][series["group_label"] == group].sum())}
    for series in all_series for group in sorted(set(series["group_label"]))
])
print(group_counts.to_string(index=False))
STEP_SECONDS["load"] = round(time.monotonic() - step_start, 2)


dataset                   build    scaling               stratum     group  rows  deaths
    sus   same_year_last_3_2011 consistent input_years_2011_2013      2011   314      32
    sus   same_year_last_3_2011 consistent input_years_2011_2013      2012   326      36
    sus   same_year_last_3_2011 consistent input_years_2011_2013      2013   334      22
    sus   same_year_last_3_2011 historical input_years_2011_2013      2011   314      32
    sus   same_year_last_3_2011 historical input_years_2011_2013      2012   326      36
    sus   same_year_last_3_2011 historical input_years_2011_2013      2013   334      22
    sus prospective_last_3_2011 consistent input_years_2011_2013      2011   274      26
    sus prospective_last_3_2011 consistent input_years_2011_2013      2012   285      12
    sus prospective_last_3_2011 consistent input_years_2011_2013      2013   304      19
flchain           horizon_1095d consistent             1997_eval 1997_eval   340      26
flchain           hor

## 3. Metric functions
Each function does one step. `layer_inputs` picks one layer's rows. `layer_metric_values` computes the curve metrics of one layer for a set of row weights (unit weights for the point value, patient-draw counts for a bootstrap draw): spectrum of both tokens and gradient metrics of the target token, with the same functions the main run uses (`lc.weighted_spectrum`, `lc.weighted_gradient_metrics`). `layer_analysis` runs the Work 2 geometry of a group, which supplies the random-direction shares. No probe is fitted here.

In [3]:
def layer_inputs(series, rows, layer):
    """One layer's arrays for the given rows: both tokens, and the target gradient as probability and as log-odds."""
    gradient = series["gradient"][rows, layer]
    probability = series["probability"][rows]
    return {
        "target": series["target"][rows, layer],
        "feature_mean": series["feature_mean"][rows, layer],
        "gradient": {"probability": gradient, "log_odds": geometry.log_odds_gradients(gradient, probability)},
        "scale_target": series["scale_target"][layer],
        "scale_feature_mean": series["scale_feature_mean"][layer],
    }


def layer_metric_values(inputs, weights):
    """Curve metrics of one layer for row weights, keyed by (token, chart, metric)."""
    spectra = {"target": (inputs["target"], inputs["scale_target"]),
               "feature_mean": (inputs["feature_mean"], inputs["scale_feature_mean"])}
    values = {}
    for token, (embeddings, scale) in spectra.items():
        for chart, block in lc.weighted_spectrum(embeddings, scale, weights).items():
            for metric in lc.SPECTRUM_METRICS:
                values[(token, chart, metric)] = block[metric]
    for kind in GRADIENT_KINDS:
        gradient_metrics = lc.weighted_gradient_metrics(inputs["gradient"][kind], inputs["scale_target"], weights)
        for (chart, metric), value in gradient_metrics.items():
            values[("target", chart, f"{kind}_{metric}")] = value
    return values


def layer_analysis(series, rows, inputs):
    """Work 2 geometry of one group at one layer; its random-direction shares are used as point values."""
    return probe.analyze_layer_geometry(
        inputs["target"], inputs["gradient"]["probability"], series["probability"][rows], inputs["scale_target"],
        rng=np.random.default_rng(DIRECTION_SEED), n_random=probe.RANDOM_DIRECTIONS,
        top_axes=probe.TOP_AXES, interior=probe.INTERIOR_CUTS)


def analysis_as_metrics(analysis):
    """Work 2 output of one layer re-keyed to the (token, chart, metric) names of layer_metric_values."""
    metrics = {}
    for chart, block in analysis["spectrum"].items():
        for metric in lc.SPECTRUM_METRICS:
            metrics[("target", chart, metric)] = block[metric]
    for (chart, kind), block in analysis["dimension"].items():
        for metric in ("centered_pr", "uncentered_pr"):
            metrics[("target", chart, f"{kind}_{metric}")] = block[metric]
    for (chart, kind), block in analysis["coherence"].items():
        metrics[("target", chart, f"{kind}_resultant_length")] = block["resultant_length"]
    return metrics


## 4. Leading direction against input year
Per series and layer: the first principal axis of the centred raw target embeddings over all pooled rows of the series, its score per row, and the Pearson correlation of that score with input year. The sign of an eigenvector is arbitrary, so the absolute correlation is reported too. `pc1_var_share` is the share of the variance on that axis.

In [4]:
step_start = time.monotonic()


def leading_direction_rows(series):
    """Per layer: variance share of the first principal axis and its correlation with input year."""
    rows = []
    for layer in range(LAYER_COUNT):
        embeddings = series["target"][:, layer]
        mean, axes, variances = geometry.principal_axes(embeddings)
        score = (embeddings - mean) @ axes[:, 0]
        correlation = float(np.corrcoef(score, series["years"])[0, 1])
        rows.append({
            "dataset": series["dataset"], "build": series["build"], "scaling": series["scaling"],
            "stratum": series["stratum"], "layer": layer,
            "pc1_var_share": float(variances[0] / variances.sum()),
            "pearson_r": correlation, "abs_pearson_r": abs(correlation), "rows": len(score),
        })
    return rows


leading = pd.DataFrame([row for series in all_series for row in leading_direction_rows(series)])
leading[leading["dataset"] == "sus"].drop(columns="dataset").to_csv(
    SUS_OUT / "leading_direction_year_correlation.csv", index=False)
leading[leading["dataset"] == "flchain"].drop(columns="dataset").to_csv(
    FLCHAIN_OUT / "leading_direction_year_correlation.csv", index=False)
print(leading[["dataset", "build", "scaling", "stratum", "layer", "pc1_var_share", "pearson_r"]].to_string(index=False))
STEP_SECONDS["leading_direction"] = round(time.monotonic() - step_start, 2)


/home/rafaelmg/Documents/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/rafaelmg/Documents/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


dataset                   build    scaling               stratum  layer  pc1_var_share  pearson_r
    sus   same_year_last_3_2011 consistent input_years_2011_2013      0       0.980026   0.993761
    sus   same_year_last_3_2011 consistent input_years_2011_2013      1       0.975091   0.988374
    sus   same_year_last_3_2011 consistent input_years_2011_2013      2       0.973045  -0.981585
    sus   same_year_last_3_2011 consistent input_years_2011_2013      3       0.959476   0.950489
    sus   same_year_last_3_2011 consistent input_years_2011_2013      4       0.905519   0.932372
    sus   same_year_last_3_2011 consistent input_years_2011_2013      5       0.909794   0.898140
    sus   same_year_last_3_2011 consistent input_years_2011_2013      6       0.931273   0.900579
    sus   same_year_last_3_2011 consistent input_years_2011_2013      7       0.903596   0.885903
    sus   same_year_last_3_2011 consistent input_years_2011_2013      8       0.846571   0.879082
    sus   same_year_

## 5. Sanity checks
(a) With every row of a series counted once, the point values of the four main-run metrics (raw and standardized embedding participation ratio, raw and standardized probability resultant length) must equal the matching rows of the main `layer_curves.csv`, at every layer, within 1e-6.

(b) For the first group of the first SUS build (year 2011, unit weights), the spectrum, the gradient participation ratios and the gradient resultant lengths (probability and log-odds) must equal the values that `analyze_layer_geometry` returns for the same rows, at every layer, within 1e-6.

In [5]:
step_start = time.monotonic()
main_curves = {
    "sus": pd.read_csv(SUS_DIR / "layer_curves.csv"),
    "flchain": pd.read_csv(FLCHAIN_DIR / "layer_curves.csv"),
}


def main_run_estimate(curves, series, token, chart, metric, layer):
    """Point estimate of one row of the main run's layer_curves.csv for this series."""
    selected = curves[(curves["build"] == series["build"]) & (curves["scaling"] == series["scaling"])
                      & (curves["stratum"] == series["stratum"]) & (curves["token"] == token)
                      & (curves["chart"] == chart) & (curves["metric"] == metric) & (curves["layer"] == layer)]
    if len(selected) != 1:
        raise ValueError(f"expected one main-run row, found {len(selected)}")
    return float(selected["estimate"].iloc[0])


def sanity_row(check, series, layer, token, chart, metric, measured, reference, source):
    """One comparison of the sanity table, with its pass or fail under SANITY_TOLERANCE."""
    passed = bool(np.isclose(measured, reference, rtol=0, atol=SANITY_TOLERANCE, equal_nan=True))
    return {"check": check, "dataset": series["dataset"], "build": series["build"], "scaling": series["scaling"],
            "stratum": series["stratum"], "layer": layer, "metric": f"{token}|{chart}|{metric}",
            "measured": measured, "reference": reference, "reference_source": source,
            "abs_difference": abs(measured - reference), "passed": passed}


sanity_rows = []
for series in all_series:
    everyone = np.arange(len(series["years"]))
    for layer in range(LAYER_COUNT):
        measured = layer_metric_values(layer_inputs(series, everyone, layer), np.ones(len(everyone)))
        for token, chart, metric in SANITY_METRICS:
            reference = main_run_estimate(main_curves[series["dataset"]], series, token, chart, metric, layer)
            sanity_rows.append(sanity_row("pooled_point_matches_main_run", series, layer, token, chart, metric,
                                          measured[(token, chart, metric)], reference, "layer_curves.csv"))

check_series = sus_all[0]
check_group = sorted(set(check_series["group_label"]))[0]
check_rows = np.flatnonzero(check_series["group_label"] == check_group)
for layer in range(LAYER_COUNT):
    inputs = layer_inputs(check_series, check_rows, layer)
    measured = layer_metric_values(inputs, np.ones(len(check_rows)))
    references = analysis_as_metrics(layer_analysis(check_series, check_rows, inputs))
    for (token, chart, metric), reference in references.items():
        sanity_rows.append(sanity_row("group_matches_analyze_layer_geometry", check_series, layer, token, chart,
                                      metric, measured[(token, chart, metric)], reference,
                                      "analyze_layer_geometry"))

sanity_table = pd.DataFrame(sanity_rows)
sanity_table[sanity_table["dataset"] == "sus"].to_csv(SUS_OUT / "sanity_checks.csv", index=False)
sanity_table[sanity_table["dataset"] == "flchain"].to_csv(FLCHAIN_OUT / "sanity_checks.csv", index=False)
summary = sanity_table.groupby(["check", "dataset", "build", "scaling", "stratum"]).agg(
    comparisons=("passed", "size"), passed=("passed", "sum"), max_abs_difference=("abs_difference", "max"))
print(summary.to_string())
if not sanity_table["passed"].all():
    raise RuntimeError(f"sanity checks failed: {int((~sanity_table['passed']).sum())}")
print(f"Sanity checks passed: {int(sanity_table['passed'].sum())}/{len(sanity_table)}")
STEP_SECONDS["sanity"] = round(time.monotonic() - step_start, 2)


                                                                                                       comparisons  passed  max_abs_difference
check                                dataset build                   scaling    stratum                                                       
group_matches_analyze_layer_geometry sus     same_year_last_3_2011   consistent input_years_2011_2013          240     240        2.664535e-15
pooled_point_matches_main_run        flchain horizon_1095d           consistent 1997_eval                       48      48        1.776357e-15
                                                                                1998-2003                       48      48        1.776357e-15
                                     sus     prospective_last_3_2011 consistent input_years_2011_2013           48      48        4.440892e-16
                                             same_year_last_3_2011   consistent input_years_2011_2013           48      48        6.661338e-16

## 6. Curves per group and layer
For each group, the point values come from all of its rows. The patient bootstrap draws the group's own patients with replacement (500 draws, seed 20261106, `prospective_audit.patient_draw_counts`); each draw recomputes the metrics at every layer. Intervals are the 2.5 and 97.5 percentiles over the draws. The difference from layer 11 is paired within each draw. Random-direction shares are point values with the direction standard error, as in the main run (no row bootstrap).

In [6]:
step_start = time.monotonic()


def compute_group(series, group):
    """Curve rows of one group: point values, bootstrap intervals, paired differences and random-direction shares."""
    rows = np.flatnonzero(series["group_label"] == group)
    group_ids = series["patient_ids"][rows]
    universe, counts = pa.patient_draw_counts(group_ids, draws=DRAWS, seed=BOOTSTRAP_SEED)
    weights = counts[:, pa.row_patient_codes(group_ids, universe)].astype(np.float64)
    inputs = [layer_inputs(series, rows, layer) for layer in range(LAYER_COUNT)]
    point = [layer_metric_values(inputs[layer], np.ones(len(rows))) for layer in range(LAYER_COUNT)]
    keys = list(point[0])
    draw_values = np.full((DRAWS, LAYER_COUNT, len(keys)), np.nan)
    for draw in range(DRAWS):
        for layer in range(LAYER_COUNT):
            metrics = layer_metric_values(inputs[layer], weights[draw])
            draw_values[draw, layer] = [metrics[key] for key in keys]
    analysis = [layer_analysis(series, rows, inputs[layer]) for layer in range(LAYER_COUNT)]
    deaths = int(series["deaths"][rows].sum())

    curve_rows = []
    for layer in range(LAYER_COUNT):
        for index, (token, chart, metric) in enumerate(keys):
            values = draw_values[:, layer, index]
            low, high, valid = lc.interval(values)
            difference_low, difference_high, _ = lc.interval(values - draw_values[:, LAST_LAYER, index])
            estimate = point[layer][(token, chart, metric)]
            curve_rows.append(lc.curve_row(
                series["stratum"], layer, token, chart, metric, estimate, low, high, valid,
                estimate - point[LAST_LAYER][(token, chart, metric)], difference_low, difference_high,
                len(rows), deaths))
        for (chart, family), block in analysis[layer]["random"].items():
            for metric in ("interior_share", "interior_se"):
                curve_rows.append(lc.curve_row(
                    series["stratum"], layer, "target", chart, f"random_{family}_{metric}", block[metric],
                    np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, len(rows), deaths))

    table = pd.DataFrame(curve_rows)
    table.insert(0, "scaling", series["scaling"])
    table.insert(0, "build", series["build"])
    table.insert(3, series["group_column"], group)
    return table


tables = {"sus": [], "flchain": []}
with threadpool_limits(limits=BOOTSTRAP_THREADS):
    for series in all_series:
        for group in sorted(set(series["group_label"])):
            group_start = time.monotonic()
            tables[series["dataset"]].append(compute_group(series, group))
            print(f"{series['build']} {series['scaling']} {series['stratum']} group {group}: "
                  f"{time.monotonic() - group_start:.1f} s")
sus_curves = pd.concat(tables["sus"], ignore_index=True)
flchain_curves = pd.concat(tables["flchain"], ignore_index=True)
sus_curves.to_csv(SUS_OUT / "layer_curves_by_year.csv", index=False)
flchain_curves.to_csv(FLCHAIN_OUT / "layer_curves_by_year.csv", index=False)
print(f"SUS rows: {len(sus_curves)} | FLChain rows: {len(flchain_curves)}")
STEP_SECONDS["curves"] = round(time.monotonic() - step_start, 2)


same_year_last_3_2011 consistent input_years_2011_2013 group 2011: 92.0 s


same_year_last_3_2011 consistent input_years_2011_2013 group 2012: 99.2 s


same_year_last_3_2011 consistent input_years_2011_2013 group 2013: 104.0 s


same_year_last_3_2011 historical input_years_2011_2013 group 2011: 97.2 s


same_year_last_3_2011 historical input_years_2011_2013 group 2012: 102.4 s


same_year_last_3_2011 historical input_years_2011_2013 group 2013: 105.2 s


... (5 outputs left out)

horizon_1095d consistent 1998-2003 group 1999: 103.8 s


horizon_1095d consistent 1998-2003 group 2000: 79.5 s


horizon_1095d consistent 1998-2003 group 2001: 67.8 s


horizon_1095d consistent 1998-2003 group 2002-2003: 86.1 s
SUS rows: 4320 | FLChain rows: 2880


## 7. Figures
One figure per metric family. Each panel is one build (SUS) or one stratum (FLChain); each line is one input-year group, with its 95% interval as a band (the random-direction share has its standard error as bars). The dotted vertical line marks layer 11.

In [7]:
step_start = time.monotonic()


def plot_family(curves, spec, group_column, panel_columns, figure_dir):
    """One figure of one metric family: a panel per build or stratum, a line per input-year group."""
    file_name, title, y_label, metric, chart, se_metric = spec
    frame = curves.assign(
        panel=curves[panel_columns].astype(str).agg(" / ".join, axis=1),
        group=curves[group_column].astype(str))
    frame = frame[(frame["token"] == "target") & (frame["chart"] == chart)]
    panels = list(dict.fromkeys(frame["panel"]))
    figure, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4), sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for axis, panel in zip(axes, panels):
        for group in sorted(frame.loc[frame["panel"] == panel, "group"].unique()):
            part = frame[(frame["panel"] == panel) & (frame["group"] == group) & (frame["metric"] == metric)]
            part = part.sort_values("layer")
            if se_metric is None:
                line, = axis.plot(part["layer"], part["estimate"], marker="o", markersize=3, label=group)
                axis.fill_between(part["layer"], part["low"], part["high"], alpha=0.15, color=line.get_color())
            else:
                spread = frame[(frame["panel"] == panel) & (frame["group"] == group)
                               & (frame["metric"] == se_metric)].sort_values("layer")
                axis.errorbar(part["layer"], part["estimate"], yerr=spread["estimate"].to_numpy(),
                              marker="o", markersize=3, capsize=2, label=group)
        axis.axvline(LAST_LAYER, color="gray", linestyle=":")
        axis.set_title(panel, fontsize=9)
        axis.set_xlabel("layer")
    axes[0].set_ylabel(y_label)
    axes[0].legend(title=group_column, fontsize=8)
    figure.suptitle(title)
    figure.savefig(figure_dir / f"{file_name}.png", dpi=120)
    plt.close(figure)


for curves, group_column, panel_columns, out_dir in (
        (sus_curves, "input_year", ["build", "scaling"], SUS_OUT),
        (flchain_curves, "sample_year_group", ["stratum"], FLCHAIN_OUT)):
    for spec in FIGURE_SPECS:
        plot_family(curves, spec, group_column, panel_columns, out_dir / "figures")
print(sorted(path.name for path in (SUS_OUT / "figures").glob("*.png")))
print(sorted(path.name for path in (FLCHAIN_OUT / "figures").glob("*.png")))
STEP_SECONDS["figures"] = round(time.monotonic() - step_start, 2)


['gradient_probability_resultant_length_raw.png', 'random_isotropic_interior_share_raw.png', 'spectrum_embedding_pr_raw.png', 'spectrum_embedding_pr_standardized.png', 'spectrum_top1_share_raw.png']
['gradient_probability_resultant_length_raw.png', 'random_isotropic_interior_share_raw.png', 'spectrum_embedding_pr_raw.png', 'spectrum_embedding_pr_standardized.png', 'spectrum_top1_share_raw.png']


## 8. Run record
Times, versions, input hashes, rows per group and the sanity result, written once per dataset.

In [8]:
def write_run_record(out_dir, dataset, series_list, input_paths):
    """Run record of one dataset: times, versions, input hashes, rows per group and the sanity result."""
    dataset_sanity = sanity_table[sanity_table["dataset"] == dataset]
    record = {
        "dataset": dataset,
        "started_utc": STARTED_UTC,
        "finished_utc": datetime.now(timezone.utc).isoformat(),
        "runtime_seconds_at_write": round(time.monotonic() - T0, 2),
        "step_seconds": STEP_SECONDS,
        "device": "cpu",
        "python": platform.python_version(),
        "versions": {name: metadata.version(name) for name in VERSIONED_PACKAGES},
        "bootstrap_draws": DRAWS,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "direction_seed": DIRECTION_SEED,
        "bootstrap_blas_thread_limit": BOOTSTRAP_THREADS,
        "sanity_checks_passed": int(dataset_sanity["passed"].sum()),
        "sanity_checks_total": int(len(dataset_sanity)),
        "inputs_sha256": {str(path): probe.file_sha256(path) for path in input_paths},
        "groups": [
            {"build": series["build"], "scaling": series["scaling"], "stratum": series["stratum"], "group": group,
             "rows": int(np.sum(series["group_label"] == group)),
             "deaths": int(series["deaths"][series["group_label"] == group].sum())}
            for series in series_list for group in sorted(set(series["group_label"]))
        ],
    }
    (out_dir / "run_record.json").write_text(json.dumps(record, indent=2, default=str))
    return record


sus_record = write_run_record(
    SUS_OUT, "sus", sus_all,
    [SUS_DIR / "layer_arrays.npz", SUS_DIR / "layer_curves.csv"] + CODE_FILES)
flchain_record = write_run_record(
    FLCHAIN_OUT, "flchain", flchain_all,
    [FLCHAIN_DIR / "layer_arrays.npz", FLCHAIN_DIR / "layer_curves.csv", FLCHAIN_CSV] + CODE_FILES)
print(f"Run completed in {round(time.monotonic() - T0, 2)} s; sanity checks "
      f"{sus_record['sanity_checks_passed'] + flchain_record['sanity_checks_passed']}/"
      f"{sus_record['sanity_checks_total'] + flchain_record['sanity_checks_total']} passed")


Run completed in 1499.54 s; sanity checks 480/480 passed


## Reading

Written by the orchestrator (2026-11-26). Sanity checks pass:
- pooled rows reproduce the main runs' point estimates: 240/240, max abs difference 2.7e-15;
- unit-weight group values equal `analyze_layer_geometry`: 240/240.

The bootstrap here resamples patients within each year group, so its intervals are not the main runs' joint draws.

**Main points.**
- In both datasets, the pooled multi-year early-layer spectrum is dominated by a direction that tracks input year.
  - SUS: |r| between the pooled leading direction and input year is 0.88–0.99 through layer 9 in all three builds, and 0.74–0.89 at layer 10. The historical build's leading direction holds only 33–40% of the variance at layers 8–10.
  - FLChain: the pooled 1998–2003 stratum has layer-0 PR 1.10. The groups 1998–2001 alone have 3.6–4.4. The `2002-2003` group (1.00) itself pools two sample years: in the final review's split each year alone has 3.35 / 3.76, and |r| with year inside the group is 1.00.
- **Embedding PR within year.** Raw PR at layer 9 exceeds layer 11 in every year group of every build:
  - SUS same-year: +1.1 to +2.4;
  - SUS prospective: +0.3 to +0.6, with one interval including 0;
  - FLChain: +0.43 to +1.04.

  Standardized: positive for SUS same-year (all intervals exclude 0) and for FLChain in 4 of 6 groups (1999 and 2001 include 0); no difference for SUS prospective.
- **Layer 10 within year.** Raw PR at layer 10 exceeds layer 11 in SUS (all same-year groups; prospective 2 of 3), but not in FLChain (at or below layer 11 in 5 of 6 groups).
- **Gradients within year.** Gradients are less concentrated at layers 9–10 than at layer 11 in every group:
  - SUS same-year: R 0.84–0.89 vs 0.96–0.97;
  - SUS prospective: 0.978–0.984 vs 0.997;
  - FLChain: about 0.93 vs 0.975.

  The random isotropic interior share is roughly twice as large at layer 9 as at layer 11.
- **Probe.** Per-year probe AUROC is open: the fitting-row arrays were not saved.
- **Layer choice.** No layer is selected here.